<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/memory/recall_memory_before_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Recall Memory Before a RAG Query

Agent memory and document retrieval solve different problems. Memory stores conversation context (preferences, prior answers, extracted facts). A vector index stores external knowledge (policies, manuals, tickets).

This notebook demonstrates a safe pattern: **recall user memory first**, then run RAG, and combine both contexts in the final answer.

In [ ]:
%pip install llama-index-core llama-index-llms-openai llama-index-embeddings-openai

In [ ]:
import os

os.environ["OPENAI_API_KEY"] = "sk-proj-..."

## Build a tiny document index (RAG corpus)

In [ ]:
from llama_index.core import Document, VectorStoreIndex
from llama_index.core.retrievers import VectorIndexRetriever

docs = [
    Document(
        text="Refund policy: annual plans are refundable within 30 days. Monthly plans are non-refundable."
    ),
    Document(
        text="Support hours: Monday-Friday 9am-5pm UTC. Emergency tickets are triaged within 4 hours."
    ),
]

index = VectorStoreIndex.from_documents(docs)
retriever = VectorIndexRetriever(index=index, similarity_top_k=2)

## Configure agent memory with a static preference block

In [ ]:
from llama_index.core.llms import ChatMessage
from llama_index.core.memory import Memory, StaticMemoryBlock

memory = Memory.from_defaults(
    session_id="rag-demo",
    token_limit=2000,
    memory_blocks=[
        StaticMemoryBlock(
            name="user_profile",
            static_content="User is on the annual plan and prefers concise bullet-point answers.",
            priority=0,
        )
    ],
)

memory.put_messages(
    [
        ChatMessage(role="user", content="I am evaluating a refund for my annual subscription."),
        ChatMessage(role="assistant", content="I can help — annual plans have a 30-day refund window."),
    ]
)

## Recall memory, then retrieve documents

In [ ]:
from llama_index.core.llms import ChatMessage as CM
from llama_index.core.prompts import PromptTemplate
from llama_index.llms.openai import OpenAI

user_question = "Can I get a refund and how fast will support respond?"

# Step 1: recall conversation memory BEFORE RAG
memory_messages = memory.get(messages=[CM(role="user", content=user_question)])
memory_text = "\n".join(m.content for m in memory_messages if m.content)

# Step 2: retrieve knowledge-base context (separate from memory)
nodes = retriever.retrieve(user_question)
doc_context = "\n\n".join(n.node.get_content() for n in nodes)

prompt = PromptTemplate(
    "You are a helpful support agent.\n\n"
    "## User memory\n{memory}\n\n"
    "## Retrieved policy docs\n{docs}\n\n"
    "## Question\n{question}\n\n"
    "Answer using both memory and docs. Respect the user's preferred format."
)

llm = OpenAI(model="gpt-4o-mini")
response = llm.complete(
    prompt.format(memory=memory_text, docs=doc_context, question=user_question)
)
print(response.text)

The model sees (1) remembered preferences and prior turns, then (2) policy documents from the vector index. Keeping these steps separate prevents document retrieval from overwriting session context and makes debugging easier.